In [ ]:
class UNetGenerator(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = nn.Conv2d(3, 64, 4, 2, 1)
        self.enc2 = nn.Conv2d(64, 128, 4, 2, 1)
        self.enc3 = nn.Conv2d(128, 256, 4, 2, 1)

        self.dec1 = nn.ConvTranspose2d(256, 128, 4, 2, 1)
        self.dec2 = nn.ConvTranspose2d(256, 64, 4, 2, 1)
        self.dec3 = nn.ConvTranspose2d(128, 3, 4, 2, 1)

        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()

    def forward(self, x):
        e1 = self.relu(self.enc1(x))
        e2 = self.relu(self.enc2(e1))
        e3 = self.relu(self.enc3(e2))

        d1 = self.relu(self.dec1(e3))
        d2 = self.relu(self.dec2(torch.cat([d1, e2], dim=1)))
        d3 = self.tanh(self.dec3(torch.cat([d2, e1], dim=1)))

        return d3

In [ ]:
G_st = UNetGenerator().to(device)
opt = optim.Adam(G_st.parameters(), lr=2e-4)

l1_loss = nn.L1Loss()

EPOCHS = 20

for epoch in range(EPOCHS):
    for x, y in temp_loader:
        x, y = x.to(device), y.to(device)

        pred = G_st(x)

        loss = l1_loss(pred, y)

        opt.zero_grad()
        loss.backward()
        opt.step()

    print(f"Epoch {epoch+1}")

    if epoch % 5 == 0:
        grid = utils.make_grid(pred[:8], nrow=4, normalize=True)
        plt.imshow(grid.permute(1,2,0).cpu())
        plt.title(f"ST-GAN Epoch {epoch}")
        plt.axis("off")
        plt.show()